In [0]:
import requests
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    to_date,
    explode,
    max as spark_max,
)
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    ArrayType,
)

In [0]:
def union_all(dfs):
    # Return None if the input list is empty
    if not dfs:
        return None
    out = dfs[0]
    # Union all DataFrames by column name, allowing missing columns to be filled with nulls
    for d in dfs[1:]:
        out = out.unionByName(d, allowMissingColumns=True)
    return out


In [0]:
# 0) Get the latest valid model year (exclude placeholder 9999)
max_year = (
    spark.table("bronze.model_years")
         .filter("model_year <> 9999")
         .select(spark_max("model_year").alias("y"))
         .collect()[0]["y"]
)

# 1) Get all unique make/model pairs for the latest model year
driver_df = (
    spark.table("bronze.vehicle_models")
         .filter(col("model_year") == lit(int(max_year)))
         .select("make", "model")
         .distinct()
)  

# Collect make/model pairs as a list of Row objects
pairs = driver_df.collect()

In [0]:
# Schema for raw NHTSA complaints API response
raw_schema = StructType([
    StructField("odiNumber", StringType(), True),  # Complaint ID
    StructField("manufacturer", StringType(), True),  # Vehicle manufacturer
    StructField("crash", BooleanType(), True),  # Crash involved (True/False)
    StructField("fire", BooleanType(), True),  # Fire involved (True/False)
    StructField("numberOfInjuries", IntegerType(), True),  # Number of injuries
    StructField("numberOfDeaths", IntegerType(), True),  # Number of deaths
    StructField("dateOfIncident", StringType(), True),  # Date of incident (MM/dd/yyyy)
    StructField("dateComplaintFiled", StringType(), True),  # Date complaint filed (MM/dd/yyyy)
    StructField("vin", StringType(), True),  # Vehicle Identification Number
    StructField("components", StringType(), True),  # Affected components (comma-separated)
    StructField("summary", StringType(), True),  # Complaint summary text
    StructField("products", ArrayType(StructType([
        StructField("type", StringType(), True),  # Product type (e.g., vehicle, tire)
        StructField("productYear", StringType(), True),  # Product year
        StructField("productMake", StringType(), True),  # Product make
        StructField("productModel", StringType(), True),  # Product model
        StructField("manufacturer", StringType(), True),  # Product manufacturer
    ])), True),  # List of products related to the complaint
])

In [0]:
# 3) acumulăm rezultate în liste (pentru demo e ok)
all_complaints = []  # List to accumulate complaint DataFrames for each make/model
all_products = []    # List to accumulate product DataFrames for each make/model
all_components = []  # List to accumulate component DataFrames for each make/model

base_url = "https://api.nhtsa.gov/complaints/complaintsByVehicle"

for r in pairs:
    make = r["make"]
    model = r["model"]
    model_year = int(max_year)

    # Call NHTSA API for each make/model/year
    resp = requests.get(base_url, params={"make": make, "model": model, "modelYear": model_year}, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    results = data.get("results", [])
    if not results:
        continue

    # Create raw DataFrame from API results using predefined schema
    df_raw = spark.createDataFrame(results, schema=raw_schema)

    # complaints (fără products)
    # Select and transform complaint fields, add context columns, and deduplicate by odi_number
    df_complaints = (
        df_raw.select(
            col("odiNumber").cast("bigint").alias("odi_number"),
            col("manufacturer").alias("manufacturer"),
            col("crash").cast("boolean").alias("crash"),
            col("fire").cast("boolean").alias("fire"),
            col("numberOfInjuries").cast("int").alias("number_of_injuries"),
            col("numberOfDeaths").cast("int").alias("number_of_deaths"),
            to_date(col("dateOfIncident"), "MM/dd/yyyy").alias("date_of_incident"),
            to_date(col("dateComplaintFiled"), "MM/dd/yyyy").alias("date_complaint_filed"),
            col("vin").alias("vin"),
            col("components").alias("components"),
            col("summary").alias("summary"),
        )
        .withColumn("model_year", lit(model_year))
        .withColumn("make", lit(make))
        .withColumn("model", lit(model))
        .withColumn("ingestion_ts", current_timestamp())
        .dropDuplicates(["odi_number"])
    )

    # products
    # Explode products array and select relevant fields, deduplicate by composite key
    df_products = (
        df_raw.select(
            col("odiNumber").cast("bigint").alias("odi_number"),
            explode(col("products")).alias("p")
        )
        .select(
            col("odi_number"),
            col("p.type").alias("product_type"),
            col("p.productYear").cast("int").alias("product_year"),
            col("p.productMake").alias("product_make"),
            col("p.productModel").alias("product_model"),
        )
        .withColumn("ingestion_ts", current_timestamp())
        .dropDuplicates(["odi_number", "product_type", "product_year", "product_make", "product_model"])
    )

    # components
    # Split components string, explode to rows, clean and deduplicate
    df_components = (
         df_raw
      .select(
          col("odiNumber").cast("bigint").alias("odi_number"),
          explode(split(col("components"), ",")).alias("component")
      )
      .select(
          col("odi_number"),
          trim(lower(col("component"))).alias("component")
      )
      .filter(col("component").isNotNull() & (col("component") != ""))
      .withColumn("ingestion_ts", current_timestamp())
      .dropDuplicates(["odi_number", "component"])
    )

    # Accumulate DataFrames for later union
    all_complaints.append(df_complaints)
    all_products.append(df_products)
    all_components.append(df_components)

In [0]:
# 2) MERGE: complaints (key = odi_number)
complaints_table = "silver.complaints"
if df_complaints_final is not None:
    if spark.catalog.tableExists(complaints_table):
        dt = DeltaTable.forName(spark, complaints_table)
        (dt.alias("t")
          .merge(df_complaints_final.alias("s"), "t.odi_number = s.odi_number")
          .whenMatchedUpdateAll()  # Update all columns if odi_number matches
          .whenNotMatchedInsertAll()  # Insert all columns if odi_number does not match
          .execute()
        )
    else:
        (df_complaints_final.write.format("delta")
          .mode("overwrite")
          .saveAsTable(complaints_table)
        )


In [0]:
# 3) MERGE: products (composite key = odi_number + product_type + product_year + product_make + product_model)
products_table = "silver.complaint_products"
if df_products_final is not None:
    if spark.catalog.tableExists(products_table):
        dtp = DeltaTable.forName(spark, products_table)
        (dtp.alias("t")
          .merge(
              df_products_final.alias("s"),
              """
              t.odi_number = s.odi_number AND
              t.product_type <=> s.product_type AND
              t.product_year <=> s.product_year AND
              t.product_make <=> s.product_make AND
              t.product_model <=> s.product_model
              """
          )
          # Update ingestion_ts if a matching row is found
          .whenMatchedUpdate(set={"ingestion_ts": "current_timestamp()"})
          # Insert all columns if no match is found
          .whenNotMatchedInsertAll()
          .execute()
        )
    else:
        # Create table if it does not exist
        (df_products_final.write.format("delta")
          .mode("overwrite")
          .saveAsTable(products_table)
        )


In [0]:
# 4) MERGE: components (key = odi_number + component)
components_table = "silver.complaint_components"
if df_components_final is not None:
    if spark.catalog.tableExists(components_table):
        dtc = DeltaTable.forName(spark, components_table)
        # Merge new component records into the Delta table using composite key
        (dtc.alias("t")
          .merge(
              df_components_final.alias("s"),
              "t.odi_number = s.odi_number AND t.component = s.component"
          )
          # Update ingestion_ts if a matching row is found
          .whenMatchedUpdate(set={"ingestion_ts": "current_timestamp()"})
          # Insert all columns if no match is found
          .whenNotMatchedInsertAll()
          .execute()
        )
    else:
        # Create table if it does not exist
        (df_components_final.write.format("delta")
          .mode("overwrite")
          .saveAsTable(components_table)
        )